# Chapter 5.4: SGLang Scheduler

> **Note: This walkthrough is based on SGLang v0.3.x architecture.** The scheduler internals evolve rapidly; always refer to the latest source code.

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the Scheduler's architecture** and its role in the SGLang runtime
2. **Master waiting queue and running batch management**
3. **Learn scheduling policies**: LPM, FCFS, Random, DFS-Weight
4. **Understand chunked prefill** and its SGLang implementation
5. **See how RadixCache enables cache-aware scheduling**
6. **Trace extend/decode batch lifecycle**
7. **Compare SGLang vs vLLM scheduler design**

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.4_scheduler.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.4_scheduler.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Scheduler Architecture Overview

The Scheduler is the **brain of SGLang's runtime**. It makes all decisions about:
- Which requests to process next
- How to batch them efficiently
- When to prefill vs decode
- How to manage GPU memory

```
┌─────────────────────────────────────────────────────────────┐
│                    SCHEDULER PROCESS                        │
│                                                             │
│  ┌─────────────────┐    ┌──────────────────────────────┐   │
│  │  Waiting Queue   │    │     Running Batch             │   │
│  │  ┌─────┐        │    │  ┌─────┐ ┌─────┐ ┌─────┐    │   │
│  │  │Req A│        │───→│  │Req D│ │Req E│ │Req F│    │   │
│  │  │Req B│        │    │  └─────┘ └─────┘ └─────┘    │   │
│  │  │Req C│        │    │                               │   │
│  │  └─────┘        │    │  Extend reqs + Decode reqs    │   │
│  └─────────────────┘    └──────────────────────────────┘   │
│           │                          │                      │
│           │     Scheduling           │     Forward Pass     │
│           │     Policy               │                      │
│           ▼                          ▼                      │
│  ┌─────────────────┐    ┌──────────────────────────────┐   │
│  │  RadixCache      │    │     TpModelWorker             │   │
│  │  (prefix match)  │    │  ┌──────────────────────┐    │   │
│  └─────────────────┘    │  │    ModelRunner         │    │   │
│                          │  │    (forward pass)      │    │   │
│  ┌─────────────────┐    │  └──────────────────────┘    │   │
│  │  MemoryPool      │    └──────────────────────────────┘   │
│  │  (GPU KV-cache)  │                                       │
│  └─────────────────┘                                       │
└─────────────────────────────────────────────────────────────┘
```

## 2. The Scheduling Event Loop

The scheduler runs a continuous event loop:

```python
# sglang/srt/managers/scheduler.py (simplified)

class Scheduler:
    def event_loop_normal(self):
        """Main event loop for normal (non-overlap) scheduling."""
        while True:
            # Step 1: Receive new requests from TokenizerManager
            recv_reqs = self.recv_requests()
            
            # Step 2: Process received requests
            # - Add to waiting queue
            # - Handle abort/flush signals
            self.process_input_requests(recv_reqs)
            
            # Step 3: Get the next batch to execute
            batch = self.get_next_batch_to_run()
            
            if batch:
                # Step 4: Run model forward pass
                result = self.run_batch(batch)
                
                # Step 5: Process outputs
                self.process_batch_result(batch, result)
            else:
                # No batch to run, sleep briefly
                time.sleep(0.001)  # 1ms
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(12, 10))ax.set_xlim(0, 12)ax.set_ylim(-0.5, 12)ax.axis('off')fig.patch.set_facecolor('white')ax.text(6, 11.5, 'SGLang Scheduling Loop', fontsize=14, fontweight='bold',        ha='center', color=DARK_GRAY)steps = [    (6, 10, "Receive New Requests\nfrom TokenizerManager", BLUE),    (6, 8.5, "Match Prefixes\nin RadixCache", GREEN),    (6, 7, "Allocate KV-Cache\nMemory (MemoryPool)", ORANGE),    (6, 5.5, "Build Batch\n(Extend + Decode reqs)", RED),    (6, 4, "Forward Pass\n(TpModelWorker -> GPU)", PURPLE),    (6, 2.5, "Process Outputs\n(sample tokens, check EOS)", BLUE),    (6, 1, "Cache Finished Reqs\nin RadixCache", GREEN),]for x, y, label, color in steps:    box = mpatches.FancyBboxPatch((x-2.2, y-0.5), 4.4, 1, boxstyle="round,pad=0.1",                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)    ax.add_patch(box)    ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')# Arrows between stepsfor i in range(len(steps)-1):    ax.annotate('', xy=(6, steps[i+1][1]+0.5), xytext=(6, steps[i][1]-0.5),                arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=2))# Loop back arrowax.annotate('', xy=(10.5, 10.5), xytext=(10.5, 1),            arrowprops=dict(arrowstyle='->', color=GRAY, lw=2,                           connectionstyle='arc3,rad=-0.3'))ax.text(11, 5.5, 'Loop', fontsize=10, fontweight='bold', color=GRAY, rotation=90, va='center')plt.tight_layout()plt.savefig("scheduling_loop.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Complete Scheduler simulation

from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from enum import Enum
import time
import random

class RequestStatus(Enum):
    WAITING = "waiting"
    RUNNING_EXTEND = "extending"  # Prefill phase
    RUNNING_DECODE = "decoding"   # Decode phase
    FINISHED = "finished"
    ABORTED = "aborted"

@dataclass
class SchedulerRequest:
    """A request being managed by the scheduler."""
    rid: str
    input_ids: List[int]
    max_output_tokens: int
    
    # Computed by scheduler
    prefix_len: int = 0          # Tokens matched in RadixCache
    extend_len: int = 0          # Tokens to prefill
    output_ids: List[int] = field(default_factory=list)
    status: RequestStatus = RequestStatus.WAITING
    arrival_time: float = 0.0
    
    @property
    def total_tokens(self):
        return len(self.input_ids) + len(self.output_ids)
    
    @property
    def is_finished(self):
        return len(self.output_ids) >= self.max_output_tokens


class SchedulerSimulator:
    """Simulates SGLang's Scheduler."""
    
    def __init__(
        self,
        max_running_requests: int = 16,
        max_total_tokens: int = 4096,
        schedule_policy: str = "lpm",
        chunk_prefill_budget: int = 0,
    ):
        self.max_running_requests = max_running_requests
        self.max_total_tokens = max_total_tokens
        self.schedule_policy = schedule_policy
        self.chunk_prefill_budget = chunk_prefill_budget
        
        # Queues
        self.waiting_queue: List[SchedulerRequest] = []
        self.running_batch: List[SchedulerRequest] = []
        self.completed: List[SchedulerRequest] = []
        
        # Resource tracking
        self.current_tokens = 0  # Tokens in KV-cache
        
        # Simulated cache (token prefix -> True)
        self.cached_prefixes: set = set()
        
        # Stats
        self.step_count = 0
        self.total_forward_passes = 0
        self.history = []
    
    def add_request(self, req: SchedulerRequest):
        req.arrival_time = time.time()
        self.waiting_queue.append(req)
    
    def _get_prefix_match(self, req: SchedulerRequest) -> int:
        """Simulate RadixCache prefix matching."""
        prefix = tuple(req.input_ids)
        for length in range(len(req.input_ids), 0, -1):
            if prefix[:length] in self.cached_prefixes:
                return length
        return 0
    
    def _apply_schedule_policy(self) -> List[SchedulerRequest]:
        """Apply scheduling policy to order waiting requests."""
        if not self.waiting_queue:
            return []
        
        if self.schedule_policy == "fcfs":
            return list(self.waiting_queue)  # First come, first served
        
        elif self.schedule_policy == "lpm":
            # Longest Prefix Match: prioritize cache hits
            scored = []
            for req in self.waiting_queue:
                prefix_len = self._get_prefix_match(req)
                scored.append((prefix_len, req))
            scored.sort(key=lambda x: x[0], reverse=True)
            return [req for _, req in scored]
        
        elif self.schedule_policy == "random":
            shuffled = list(self.waiting_queue)
            random.shuffle(shuffled)
            return shuffled
        
        return list(self.waiting_queue)
    
    def get_next_batch(self) -> Optional[List[SchedulerRequest]]:
        """Build the next batch of requests to execute."""
        # Collect decode requests (already running)
        decode_reqs = [r for r in self.running_batch if r.status == RequestStatus.RUNNING_DECODE]
        
        # Token budget: how many tokens can we add?
        decode_tokens = len(decode_reqs)  # 1 token per decode request
        remaining_budget = self.max_total_tokens - self.current_tokens - decode_tokens
        remaining_slots = self.max_running_requests - len(decode_reqs)
        
        # Schedule new requests from waiting queue
        ordered = self._apply_schedule_policy()
        extend_reqs = []
        
        for req in ordered:
            if remaining_slots <= 0:
                break
            
            prefix_len = self._get_prefix_match(req)
            extend_tokens = len(req.input_ids) - prefix_len
            
            if extend_tokens <= remaining_budget:
                req.prefix_len = prefix_len
                req.extend_len = extend_tokens
                req.status = RequestStatus.RUNNING_EXTEND
                extend_reqs.append(req)
                remaining_budget -= extend_tokens
                remaining_slots -= 1
                self.current_tokens += len(req.input_ids)
        
        # Remove scheduled requests from waiting queue
        for req in extend_reqs:
            self.waiting_queue.remove(req)
        
        batch = decode_reqs + extend_reqs
        return batch if batch else None
    
    def run_step(self):
        """Execute one scheduling step."""
        self.step_count += 1
        
        # Get batch
        batch = self.get_next_batch()
        if not batch:
            return
        
        self.total_forward_passes += 1
        self.running_batch = batch
        
        extend_count = sum(1 for r in batch if r.status == RequestStatus.RUNNING_EXTEND)
        decode_count = sum(1 for r in batch if r.status == RequestStatus.RUNNING_DECODE)
        
        # Simulate forward pass: generate one token per request
        finished_this_step = []
        for req in batch:
            req.output_ids.append(random.randint(1, 1000))
            req.status = RequestStatus.RUNNING_DECODE
            self.current_tokens += 1
            
            # Cache the prefix
            self.cached_prefixes.add(tuple(req.input_ids))
            
            if req.is_finished:
                finished_this_step.append(req)
        
        # Remove finished requests
        for req in finished_this_step:
            req.status = RequestStatus.FINISHED
            self.running_batch.remove(req)
            self.completed.append(req)
            self.current_tokens -= req.total_tokens
        
        # Record history
        self.history.append({
            "step": self.step_count,
            "extend": extend_count,
            "decode": decode_count,
            "finished": len(finished_this_step),
            "waiting": len(self.waiting_queue),
            "running": len(self.running_batch),
            "tokens": self.current_tokens,
        })

# Run a simulation
scheduler = SchedulerSimulator(
    max_running_requests=8,
    max_total_tokens=2000,
    schedule_policy="lpm"
)

# Add requests
for i in range(15):
    input_len = random.randint(20, 100)
    output_len = random.randint(5, 20)
    req = SchedulerRequest(
        rid=f"req-{i:03d}",
        input_ids=list(range(input_len)),
        max_output_tokens=output_len,
    )
    scheduler.add_request(req)

print("Scheduler Simulation")
print(f"Policy: {scheduler.schedule_policy}")
print(f"Max batch: {scheduler.max_running_requests}, Max tokens: {scheduler.max_total_tokens}")
print(f"Requests: {len(scheduler.waiting_queue)}")
print("=" * 80)
print(f"{'Step':>5s} {'Extend':>7s} {'Decode':>7s} {'Finish':>7s} {'Wait':>6s} {'Run':>5s} {'Tokens':>7s}")
print("-" * 80)

for _ in range(40):
    scheduler.run_step()
    if scheduler.history:
        h = scheduler.history[-1]
        print(f"{h['step']:5d} {h['extend']:7d} {h['decode']:7d} {h['finished']:7d} "
              f"{h['waiting']:6d} {h['running']:5d} {h['tokens']:7d}")
    if not scheduler.waiting_queue and not scheduler.running_batch:
        break

print(f"\nTotal steps: {scheduler.step_count}")
print(f"Total forward passes: {scheduler.total_forward_passes}")
print(f"Completed: {len(scheduler.completed)}")

## 3. Scheduling Policies

SGLang supports multiple scheduling policies:

### 3.1 LPM (Longest Prefix Match) — Default

Prioritizes requests that have the longest match in the RadixCache.

**Rationale**: Requests with longer cache hits require less compute.

### 3.2 FCFS (First Come First Served)

Simple FIFO ordering. Good for fairness.

### 3.3 Random

Random ordering. Useful as a baseline.

### 3.4 DFS-Weight

Depth-first search with weights. Prioritizes requests from the same tree branch to maximize cache locality.

```python
# sglang/srt/managers/schedule_policy.py (simplified)

class SchedulePolicy:
    def __init__(self, policy: str, tree_cache: RadixCache):
        self.policy = policy
        self.tree_cache = tree_cache
    
    def get_priority(self, req: Request) -> float:
        if self.policy == "lpm":
            # Higher priority for longer prefix matches
            _, prefix_len = self.tree_cache.match_prefix(req.input_ids)
            return prefix_len
        elif self.policy == "fcfs":
            return -req.arrival_time  # Older = higher priority
        elif self.policy == "random":
            return random.random()
        elif self.policy == "dfs-weight":
            # Weight based on tree depth and branch sharing
            return self._dfs_weight(req)
    
    def sort_requests(self, waiting_queue):
        return sorted(waiting_queue, 
                      key=lambda r: self.get_priority(r),
                      reverse=True)
```

In [ ]:
# Compare scheduling policies on the same workload

def run_scheduling_experiment(policy: str, requests_template: list, seed=42):
    """Run a complete scheduling experiment with given policy."""
    random.seed(seed)
    
    scheduler = SchedulerSimulator(
        max_running_requests=8,
        max_total_tokens=3000,
        schedule_policy=policy
    )
    
    for i, (input_ids, max_out) in enumerate(requests_template):
        req = SchedulerRequest(
            rid=f"req-{i:03d}",
            input_ids=list(input_ids),
            max_output_tokens=max_out,
        )
        scheduler.add_request(req)
    
    while scheduler.waiting_queue or scheduler.running_batch:
        scheduler.run_step()
        if scheduler.step_count > 200:  # Safety limit
            break
    
    return scheduler

# Create workload with prefix sharing opportunities
system_prompt = list(range(1, 51))  # 50 tokens
requests_template = []

for i in range(20):
    if i < 10:
        # Requests sharing system prompt
        user_tokens = list(range(1000 + i*20, 1000 + i*20 + random.randint(10, 30)))
        requests_template.append((system_prompt + user_tokens, random.randint(5, 15)))
    else:
        # Unique requests
        unique_tokens = list(range(2000 + i*50, 2000 + i*50 + random.randint(30, 60)))
        requests_template.append((unique_tokens, random.randint(5, 15)))

print("Scheduling Policy Comparison")
print("=" * 70)
print(f"Workload: {len(requests_template)} requests (10 with shared prefix, 10 unique)")
print()

results = {}
for policy in ["fcfs", "lpm", "random"]:
    sched = run_scheduling_experiment(policy, requests_template)
    results[policy] = sched
    
    avg_wait = 0  # simplified
    print(f"  {policy.upper():>8s}: {sched.step_count:3d} steps, "
          f"{sched.total_forward_passes:3d} forward passes, "
          f"{len(sched.completed):2d} completed")

## 4. Token Budget and Batch Size Management

The scheduler must balance multiple constraints:

```python
# sglang/srt/managers/scheduler.py — get_next_batch_to_run() (simplified)

def get_next_batch_to_run(self):
    """Build the next batch respecting all constraints."""
    
    # Constraint 1: Max batch size (number of requests)
    # Limits number of concurrent sequences
    max_bs = self.max_running_requests
    
    # Constraint 2: Max total tokens in KV-cache
    # Cannot exceed GPU memory for KV-cache
    available_tokens = self.max_total_tokens - self.current_kv_tokens
    
    # Constraint 3: Token budget for this step
    # Limits total tokens processed in one forward pass
    if self.chunk_prefill_budget > 0:
        token_budget = self.chunk_prefill_budget
    else:
        token_budget = float('inf')  # No limit
    
    # --- Build batch ---
    
    # First: include all decode requests (1 token each)
    decode_reqs = [r for r in self.running_batch 
                   if r.status == DECODE]
    batch = list(decode_reqs)
    token_budget -= len(decode_reqs)  # 1 token per decode
    available_slots = max_bs - len(decode_reqs)
    
    # Second: add extend (prefill) requests
    for req in self._sorted_waiting_queue():
        if available_slots <= 0:
            break
        
        # Check cache for prefix match
        _, prefix_len = self.radix_cache.match_prefix(req.input_ids)
        extend_tokens = len(req.input_ids) - prefix_len
        
        # Check constraints
        total_tokens_needed = len(req.input_ids)  # Full seq in KV
        
        if (extend_tokens <= token_budget and 
            total_tokens_needed <= available_tokens):
            batch.append(req)
            token_budget -= extend_tokens
            available_tokens -= total_tokens_needed
            available_slots -= 1
    
    return batch
```

In [ ]:
# Demonstrate token budget management

class TokenBudgetDemo:
    """Visualize how token budgets constrain scheduling."""
    
    def __init__(self, max_batch_size, max_total_tokens, chunk_budget=0):
        self.max_batch_size = max_batch_size
        self.max_total_tokens = max_total_tokens
        self.chunk_budget = chunk_budget
    
    def schedule(self, decode_reqs, extend_candidates):
        """Show the scheduling decision process."""
        print(f"\nConstraints:")
        print(f"  Max batch size: {self.max_batch_size}")
        print(f"  Max total tokens: {self.max_total_tokens}")
        print(f"  Chunk prefill budget: {self.chunk_budget if self.chunk_budget > 0 else 'unlimited'}")
        
        # Decode requests (always included)
        decode_tokens = len(decode_reqs)  # 1 token each
        decode_kv_tokens = sum(r["total_tokens"] for r in decode_reqs)
        
        print(f"\nDecode requests: {len(decode_reqs)} (using {decode_kv_tokens} KV tokens)")
        
        remaining_slots = self.max_batch_size - len(decode_reqs)
        remaining_kv = self.max_total_tokens - decode_kv_tokens
        remaining_budget = (self.chunk_budget - decode_tokens) if self.chunk_budget > 0 else float('inf')
        
        print(f"\nRemaining capacity:")
        print(f"  Slots: {remaining_slots}")
        print(f"  KV tokens: {remaining_kv}")
        print(f"  Prefill budget: {remaining_budget}")
        
        # Try to schedule extend requests
        scheduled = []
        rejected = []
        
        for req in extend_candidates:
            if remaining_slots <= 0:
                rejected.append((req, "batch full"))
                continue
            
            extend_tokens = req["extend_tokens"]
            total_tokens = req["total_tokens"]
            
            if total_tokens > remaining_kv:
                rejected.append((req, "KV memory full"))
                continue
            
            if extend_tokens > remaining_budget:
                rejected.append((req, "prefill budget exceeded"))
                continue
            
            scheduled.append(req)
            remaining_slots -= 1
            remaining_kv -= total_tokens
            remaining_budget -= extend_tokens
        
        print(f"\nScheduled {len(scheduled)} extend requests:")
        for req in scheduled:
            print(f"  {req['name']}: extend={req['extend_tokens']}, total_kv={req['total_tokens']}")
        
        if rejected:
            print(f"\nRejected {len(rejected)} requests:")
            for req, reason in rejected:
                print(f"  {req['name']}: {reason}")

# Demo
demo = TokenBudgetDemo(
    max_batch_size=8,
    max_total_tokens=2000,
    chunk_budget=512
)

decode_reqs = [
    {"name": "decode-1", "total_tokens": 300},
    {"name": "decode-2", "total_tokens": 250},
    {"name": "decode-3", "total_tokens": 400},
]

extend_candidates = [
    {"name": "extend-A", "extend_tokens": 100, "total_tokens": 150},
    {"name": "extend-B", "extend_tokens": 200, "total_tokens": 300},
    {"name": "extend-C", "extend_tokens": 50, "total_tokens": 100},
    {"name": "extend-D", "extend_tokens": 500, "total_tokens": 600},
    {"name": "extend-E", "extend_tokens": 80, "total_tokens": 120},
]

demo.schedule(decode_reqs, extend_candidates)

## 5. Chunked Prefill

**Problem**: Long prompts can cause latency spikes because prefill blocks all decode requests.

**Solution**: Break long prefills into **chunks** and interleave with decode steps.

```
Without chunked prefill:
Step 1: [Prefill 2048 tokens]          <- All decode blocked!
Step 2: [Decode all running requests]
Step 3: [Decode all running requests]

With chunked prefill (budget=512):
Step 1: [Prefill 512 tokens + Decode]   <- Decode not blocked
Step 2: [Prefill 512 tokens + Decode]
Step 3: [Prefill 512 tokens + Decode]
Step 4: [Prefill 512 tokens + Decode]
```

```python
# sglang/srt/managers/scheduler.py — chunked prefill (simplified)

def build_extend_batch_with_chunking(self, req, budget):
    """Handle chunked prefill for a request."""
    
    extend_len = len(req.input_ids) - req.prefix_len - req.already_computed
    
    if extend_len <= budget:
        # Fits in one chunk — normal prefill
        return extend_len
    else:
        # Too long — take only a chunk
        chunk_size = budget
        req.already_computed += chunk_size
        # Request stays in "chunked prefill" state
        # Will continue in next step
        return chunk_size
```

In [ ]:
# Demonstrate chunked prefill impact on latency

class ChunkedPrefillDemo:
    """Show the difference between chunked and non-chunked prefill."""
    
    def simulate(self, prefill_tokens: int, chunk_budget: int, 
                 num_decode_reqs: int, tokens_per_ms: float = 10):
        """Simulate timing with/without chunked prefill."""
        
        if chunk_budget == 0:
            # No chunking: entire prefill in one step
            steps = []
            # Step 1: Full prefill (no decode)
            prefill_time = prefill_tokens / tokens_per_ms
            steps.append({
                "type": "prefill",
                "tokens": prefill_tokens,
                "time_ms": prefill_time,
                "decode_blocked": True
            })
            # Subsequent decode steps
            for i in range(3):
                decode_time = num_decode_reqs / tokens_per_ms
                steps.append({
                    "type": "decode",
                    "tokens": num_decode_reqs,
                    "time_ms": decode_time,
                    "decode_blocked": False
                })
            return steps
        else:
            # Chunked: interleave prefill chunks with decode
            steps = []
            remaining = prefill_tokens
            while remaining > 0:
                chunk = min(remaining, chunk_budget)
                total_tokens = chunk + num_decode_reqs
                step_time = total_tokens / tokens_per_ms
                steps.append({
                    "type": "mixed",
                    "prefill_tokens": chunk,
                    "decode_tokens": num_decode_reqs,
                    "time_ms": step_time,
                    "decode_blocked": False
                })
                remaining -= chunk
            return steps

demo = ChunkedPrefillDemo()

print("Chunked Prefill Impact Analysis")
print("=" * 70)

prefill_tokens = 2048
decode_reqs = 16
tokens_per_ms = 50  # ~50 tokens/ms throughput

for chunk_budget in [0, 512, 256]:
    label = "No chunking" if chunk_budget == 0 else f"Chunk={chunk_budget}"
    steps = demo.simulate(prefill_tokens, chunk_budget, decode_reqs, tokens_per_ms)
    
    print(f"\n--- {label} ---")
    total_time = 0
    max_decode_gap = 0
    last_decode_time = 0
    
    for i, step in enumerate(steps):
        total_time += step["time_ms"]
        if step.get("decode_blocked"):
            gap = step["time_ms"]
            max_decode_gap = max(max_decode_gap, gap)
        
        if step["type"] == "prefill":
            print(f"  Step {i+1}: PREFILL {step['tokens']} tokens "
                  f"({step['time_ms']:.1f}ms) [decode BLOCKED]")
        elif step["type"] == "mixed":
            print(f"  Step {i+1}: CHUNK {step['prefill_tokens']} + "
                  f"DECODE {step['decode_tokens']} ({step['time_ms']:.1f}ms)")
        else:
            print(f"  Step {i+1}: DECODE {step['tokens']} ({step['time_ms']:.1f}ms)")
    
    print(f"  Total steps: {len(steps)}, Max decode gap: {max_decode_gap:.1f}ms")

## 6. Extend vs Decode Batches

SGLang manages two types of operations in each batch:

### Extend (Prefill)
- Processes **multiple tokens per request** (the prompt)
- Uses **ragged tensors** (different sequence lengths)
- More compute-intensive
- FlashInfer: `BatchPrefillWithPagedKVCache`

### Decode
- Processes **one token per request** (autoregressive)
- All sequences process exactly 1 token
- More memory-bandwidth intensive
- FlashInfer: `BatchDecodeWithPagedKVCache`

```python
# sglang/srt/model_executor/forward_batch_info.py (simplified)

@dataclass
class ForwardBatch:
    """Contains all info needed for a forward pass."""
    
    # Mode
    forward_mode: ForwardMode  # EXTEND or DECODE
    
    # Token IDs to process
    input_ids: torch.Tensor     # [total_tokens]
    
    # Positions
    positions: torch.Tensor     # [total_tokens]
    
    # For extend: sequence boundaries
    extend_seq_lens: List[int]  # Length of each sequence's extend portion
    extend_prefix_lens: List[int]  # Cached prefix length
    
    # KV-cache indices
    req_pool_indices: torch.Tensor  # Which pool slots for each request
    
    # Attention metadata (for FlashInfer)
    attn_metadata: AttentionMetadata
```

In [ ]:
# Visualize extend vs decode batch structure

def visualize_batch(batch_type, requests):
    """Visualize the structure of a batch."""
    print(f"\n{batch_type} Batch")
    print("=" * 60)
    
    total_tokens = 0
    
    for req in requests:
        if batch_type == "Extend":
            prefix = req.get("prefix_len", 0)
            extend = req.get("extend_len", 0)
            total = prefix + extend
            
            cached = "C" * prefix
            new = "N" * extend
            bar = f"[{cached}{new}]"
            if len(bar) > 50:
                bar = f"[{'C'*min(prefix,20)}...{'N'*min(extend,20)}...]"
            
            print(f"  {req['name']}: {bar}")
            print(f"    Prefix (cached): {prefix} tokens, Extend (compute): {extend} tokens")
            total_tokens += extend
        else:  # Decode
            context = req.get("context_len", 0)
            print(f"  {req['name']}: [{'K'*min(context,30)}...] + [N]")
            print(f"    Context: {context} tokens, Decode: 1 token")
            total_tokens += 1
    
    print(f"\n  Total tokens in forward pass: {total_tokens}")
    print(f"  Batch size: {len(requests)} requests")

# Extend batch example
extend_requests = [
    {"name": "Req-A", "prefix_len": 50, "extend_len": 100},
    {"name": "Req-B", "prefix_len": 50, "extend_len": 30},
    {"name": "Req-C", "prefix_len": 0, "extend_len": 200},
]
visualize_batch("Extend", extend_requests)

# Decode batch example
decode_requests = [
    {"name": "Req-D", "context_len": 300},
    {"name": "Req-E", "context_len": 150},
    {"name": "Req-F", "context_len": 500},
    {"name": "Req-G", "context_len": 200},
]
visualize_batch("Decode", decode_requests)

## 7. Source Code Walkthrough: Key Scheduler Methods

```python
# sglang/srt/managers/scheduler.py — detailed walkthrough

class Scheduler:
    
    def process_input_requests(self, recv_reqs):
        """Process newly received requests."""
        for req in recv_reqs:
            if isinstance(req, TokenizedGenerateReqInput):
                # New generation request
                self.handle_generate_request(req)
            elif isinstance(req, AbortReq):
                # Abort a running request
                self.abort_request(req.rid)
            elif isinstance(req, FlushCacheReq):
                # Flush the RadixCache
                self.flush_cache()
    
    def handle_generate_request(self, req):
        """Process a new tokenized generation request."""
        # 1. Match prefix in RadixCache
        last_node, prefix_len = self.tree_cache.match_prefix(
            req.input_ids
        )
        
        # 2. Create internal request object
        internal_req = Req(
            rid=req.rid,
            input_ids=req.input_ids,
            sampling_params=req.sampling_params,
            prefix_indices=last_node.value[:prefix_len],
            last_node=last_node,
        )
        
        # 3. Lock the prefix node (prevent eviction)
        self.tree_cache.inc_lock_ref(last_node)
        
        # 4. Add to waiting queue
        self.waiting_queue.append(internal_req)
    
    def process_batch_result(self, batch, result):
        """Process output from model forward pass."""
        
        for i, req in enumerate(batch.reqs):
            # Get the sampled token
            new_token = result.output_ids[i]
            req.output_ids.append(new_token)
            
            # Check stopping criteria
            if self.should_stop(req, new_token):
                # Request is done!
                self.finish_request(req)
            else:
                # Continue decoding
                req.status = DECODE
    
    def finish_request(self, req):
        """Handle a completed request."""
        # 1. Cache the KV for future reuse
        self.tree_cache.cache_finished_req(req)
        
        # 2. Send output to DetokenizerManager
        self.send_to_detokenizer(req)
        
        # 3. Remove from running batch
        self.running_batch.remove(req)
        
        # 4. Free memory that's not cached
        self.tree_cache.dec_lock_ref(req.last_node)
```

In [ ]:
# Detailed scheduler state tracking

class DetailedSchedulerTrace:
    """Provides detailed traces of scheduler decisions."""
    
    def __init__(self):
        self.events = []
    
    def log(self, step, event_type, details):
        self.events.append((step, event_type, details))
    
    def trace_scheduling_decision(self, step, waiting, running, 
                                  selected, rejected_reasons):
        """Log a detailed scheduling decision."""
        self.log(step, "SCHEDULE", {
            "waiting_count": len(waiting),
            "running_count": len(running),
            "selected": [r.rid for r in selected],
            "rejected": rejected_reasons,
        })
    
    def display(self, max_events=20):
        print("\nScheduler Decision Trace")
        print("=" * 70)
        for step, event_type, details in self.events[:max_events]:
            print(f"  Step {step:3d} [{event_type:10s}]")
            if isinstance(details, dict):
                for k, v in details.items():
                    print(f"    {k}: {v}")

# Example trace
trace = DetailedSchedulerTrace()

trace.log(1, "RECEIVE", {"new_requests": ["req-001", "req-002", "req-003"]})
trace.log(1, "CACHE_HIT", {"req-001": "50/80 tokens", "req-002": "0/120 tokens", "req-003": "50/60 tokens"})
trace.log(1, "SCHEDULE", {
    "policy": "LPM",
    "order": ["req-003 (83% hit)", "req-001 (63% hit)", "req-002 (0% hit)"],
    "selected": ["req-003", "req-001"],
    "rejected": {"req-002": "token budget exceeded (120 > 80 remaining)"},
})
trace.log(1, "FORWARD", {"extend": 2, "decode": 0, "total_tokens": 40})
trace.log(2, "FORWARD", {"extend": 1, "decode": 2, "total_tokens": 122})
trace.log(3, "FINISH", {"completed": "req-003", "output_tokens": 15})
trace.log(3, "CACHE", {"cached": "req-003 full sequence (75 tokens) inserted into RadixCache"})

trace.display()

## 8. Comparison with vLLM Scheduler

| Aspect | SGLang Scheduler | vLLM Scheduler |
|--------|-----------------|----------------|
| **Process Model** | Separate process | Part of AsyncEngine |
| **Scheduling Policy** | LPM (cache-aware) | Priority-based |
| **Preemption** | Historically no (LRU eviction instead); v0.4+ adds limited preemption for priority scheduling | Yes (swap/recompute) |
| **Prefix Caching** | Integral (RadixCache) | Optional (APC) |
| **Batch Types** | Extend + Decode | Prefill + Decode |
| **Chunked Prefill** | Token budget | Separate prefill/decode |
| **Memory Management** | Token-level pool | Block-level |
| **Queue Management** | Single waiting queue | Waiting + Swapped queues |
| **Fairness** | Policy-dependent | Priority + aging |

In [ ]:
# Architecture comparison visualization

print("SGLang vs vLLM Scheduler Architecture")
print("=" * 80)

print("""
SGLang Scheduler:                    vLLM Scheduler:
                                     
┌────────────────────┐              ┌────────────────────┐
│   Waiting Queue    │              │   Waiting Queue    │
│ (sorted by policy) │              │  (FIFO + priority) │
└────────┬───────────┘              └────────┬───────────┘
         │                                   │
    LPM/FCFS/Random                    Priority-based
         │                                   │
┌────────▼───────────┐              ┌────────▼───────────┐
│   Running Batch    │              │   Running Batch    │
│ Extend + Decode    │              │  Prefill + Decode  │
│ (mixed in one pass)│              │  (may be separate) │
└────────┬───────────┘              └────────┬───────────┘
         │                                   │
    Historically no                    Preemption
    preemption                      (swap/recompute)
    (LRU eviction;
     v0.4+ adds limited
     preemption support)
         │                                   │
┌────────▼───────────┐              ┌────────▼───────────┐
│   RadixCache       │              │   BlockManager     │
│ (radix tree)       │              │  (block table)     │
│ Token-level        │              │  Block-level       │
│ Auto prefix share  │              │  Hash-based APC    │
└────────────────────┘              └────────────────────┘
                                    ┌────────────────────┐
                                    │   Swapped Queue    │
                                    │  (preempted reqs)  │
                                    └────────────────────┘
""")

print("Key Differences:")
print("  1. SGLang historically did not use preemption (relying on LRU eviction instead),")
print("     though newer versions (v0.4+) have added limited preemption support for priority-based scheduling.")
print("  2. SGLang's LPM policy is cache-aware by default")
print("  3. vLLM has a swapped queue for preempted requests")
print("  4. SGLang mixes extend+decode in same forward pass")
print("  5. vLLM uses block-level memory, SGLang uses token-level")

## 9. Summary

### Key Takeaways

1. **The Scheduler is the brain** of SGLang's runtime, making all batch composition decisions
2. **LPM (Longest Prefix Match)** is the default policy, maximizing cache reuse
3. **Token budget** constrains how many tokens can be processed per step
4. **Chunked prefill** prevents long prompts from blocking decode requests
5. **Extend + Decode** are mixed in the same forward pass for efficiency
6. **Preemption** — SGLang historically did not use preemption (relying on LRU eviction instead), though newer versions (v0.4+) have added limited preemption support for priority-based scheduling.
7. **Cache-aware scheduling** is tightly integrated with RadixCache

### Next Chapter

Chapter 5.5 will explore **TokenizerManager & DetokenizerManager** — how tokenization is handled efficiently in separate processes.

## Exercises

### Exercise 1: Implement DFS-Weight Policy
Implement the DFS-Weight scheduling policy that prioritizes requests from the same radix tree branch.

### Exercise 2: Adaptive Chunking
Design an adaptive chunked prefill strategy that adjusts chunk size based on:
- Number of decode requests waiting
- Current GPU utilization
- Average decode latency

### Exercise 3: Scheduling Analysis
Create a workload generator and measure:
- Average time-to-first-token (TTFT) under different policies
- Token throughput (tokens/sec)
- Fairness (variance in wait time)

### Exercise 4: Preemption Design
Design a preemption mechanism for SGLang's scheduler. Consider:
- When should a request be preempted?
- How to handle preempted KV-cache in the RadixTree?
- What's the overhead vs. LRU eviction?

In [ ]:
# Exercise 1 starter: DFS-Weight policy

def dfs_weight_priority(req, radix_cache):
    """Calculate DFS-Weight priority for a request.
    
    The idea: prioritize requests that share the same tree branch,
    creating a depth-first traversal pattern that maximizes cache locality.
    
    Weight = depth_in_tree * branch_sharing_factor
    """
    # TODO: Implement
    # 1. Find the matching node in the radix tree
    # 2. Calculate depth of that node
    # 3. Count how many other waiting requests share this branch
    # 4. Return weighted score
    pass

print("Exercise 1: Implement dfs_weight_priority() above")